<a href="https://colab.research.google.com/github/Danial-071/Text-to-Speech-/blob/main/Text_to_Speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install langchain-openrouter
!pip install langchain_community
!pip install langchain_core
!pip install langchain_classic

In [ ]:
import requests

# URL of the audio file to be downloaded
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/hTqGqoC-LrW6S79HjuJUkg/trimmed-02.wav"

# Send a GET request to the URL to download the file
response = requests.get(url)

# Define the local file path where the audio file will be saved
audio_file_path = "sample-meeting.wav"

# Check if the request was successful (status code 200)
if response.status_code == 200:
	# If successful, write the content to the specified local file path
	with open(audio_file_path, "wb") as file:
		file.write(response.content)
		print("File downloaded successfully")
else:
	# If the request failed, print an error message
	print("Failed to download the file")


File downloaded successfully


In [ ]:

from transformers import pipeline

# Initialize the speech-to-text pipeline from Hugging Face Transformers
# This uses the "openai/whisper-tiny.en" model for automatic speech recognition (ASR)
# The `chunk_length_s` parameter specifies the chunk length in seconds for processing
pipe = pipeline(
  "automatic-speech-recognition",
  model="openai/whisper-tiny.en",
  chunk_length_s=30,
)

# Define the path to the audio file that needs to be transcribed
sample = 'sample-meeting.wav'

# Perform speech recognition on the audio file
# The `batch_size=8` parameter indicates how many chunks are processed at a time
# The result is stored in `prediction` with the key "text" containing the transcribed text
prediction = pipe(sample, batch_size=8)["text"]

# Print the transcribed text to the console
print(prediction)


Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_sp

 We have adopted a conservative approach to managing our leverage and have a healthy tier 1 capital ratio of 12.5%. Our forecast for the coming quarter is positive. We expect revenue to be around 135 million and 8% quarter over quarter growth driven primarily by our cutting its blockchain chain solutions and AI driven predictive analytics. We're also excited about the upcoming IPO of our FinTech subsidiary pay plus which we expect to raise 200 million significantly bolstering our liquidity and paving the way for aggressive growth strategies. We thank our shareholders for their continued faith in us and we look forward to an even more successful Q3. Thank you so much.


In [ ]:
import os
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model = 'nvidia/nemotron-3-ultra-550b-a55b:free')


In [ ]:
import torch
from transformers import pipeline
import gradio as gr

# Function to transcribe audio using the OpenAI Whisper model
def transcript_audio(audio_file):

    pipe = pipeline(model='nvidia/nemotron-3-ultra-550b-a55b:freem', chunk_length_s = 30)

    # Pass the audio_file from Gradio directly into the pipeline
    raw = pipe(audio_file, batch_size=8)

    # The pipeline returns a dictionary. Extract the text string.
    transcript = raw["text"]

    # Transcribe the audio file and return the result
    return transcript


In [ ]:
def remove_non_ascii(text):
    return ''.join(i for i in text if ord(i) < 128)


In [ ]:
def product_assistant(ascii_transcript):

    system_prompt = """You are an intelligent assistant specializing in financial products;
    your task is to process transcripts of earnings calls, ensuring that all references to
     financial products and common financial terms are in the correct format. For each
     financial product or common term that is typically abbreviated as an acronym, the full term
    should be spelled out followed by the acronym in parentheses. For example, '401k' should be
     transformed to '401(k) retirement savings plan', 'HSA' should be transformed to 'Health Savings Account (HSA)' , 'ROA' should be transformed to 'Return on Assets (ROA)', 'VaR' should be transformed to 'Value at Risk (VaR)', and 'PB' should be transformed to 'Price to Book (PB) ratio'. Similarly, transform spoken numbers representing financial products into their numeric representations, followed by the full name of the product in parentheses. For instance, 'five two nine' to '529 (Education Savings Plan)' and 'four zero one k' to '401(k) (Retirement Savings Plan)'. However, be aware that some acronyms can have different meanings based on the context (e.g., 'LTV' can stand for 'Loan to Value' or 'Lifetime Value'). You will need to discern from the context which term is being referred to  and apply the appropriate transformation. In cases where numerical figures or metrics are spelled out but do not represent specific financial products (like 'twenty three percent'), these should be left as is. Your role is to analyze and adjust financial product terminology in the text. Once you've done that, produce the adjusted transcript and a list of the words you've changed"""

    # Concatenate the system prompt and the user transcript
    prompt_input = system_prompt + "\n" + ascii_transcript

    # Create a messages object
    messages = [
        {
            "role": "user",
            "content": prompt_input
        }
    ]

    # Send the input messages to the model and retrieve its response
    response = model.invoke(messages=messages)

    # Extract and return the content of the model's first response choice
    return response['choices'][0]['message']['content']


In [ ]:
# Define the prompt template
from langchain_classic.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
template = """
Generate meeting minutes and a list of tasks based on the provided context.

Context:
{context}

Meeting Minutes:
- Key points discussed
- Decisions made

Task List:
- Actionable items with assignees and deadlines
"""
prompt = ChatPromptTemplate.from_template(template)

# Define the chain
chain = (
    {"context": RunnablePassthrough()}  # Pass the transcript as context
    | prompt
    | model
    | StrOutputParser()
)


In [ ]:
import gradio as gr

def final_function(audio_file):
    # 1. Run your pipeline
    transcript = transcript_audio(audio_file)
    transcript = remove_non_ascii(transcript)
    adjusted_file = product_assistant(transcript)
    resp = chain.invoke({'context': adjusted_file})

    # 2. Save the response to a text file for the user to download
    output_filename = "/content/meeting_minutes.txt"
    with open(output_filename, "w") as f:
        f.write(resp)

    # 3. Return BOTH the text (for the Textbox) and the file path (for the File download)
    return resp, output_filename

# Use the absolute path /content/... to avoid the InvalidPathError
audio_input = gr.Audio(sources="upload", type="filepath", label="Upload your audio file", value='/content/sample_meeting.wav')
output_text = gr.Textbox(label="Meeting Minutes and Tasks")
download_file = gr.File(label="Download the Generated Meeting Minutes and Tasks")

# Shutdown old background servers just in case
gr.close_all()

iface = gr.Interface(
    fn=final_function,  # <--- FIXED: Now points to your complete pipeline
    inputs=audio_input,
    outputs=[output_text, download_file], # Expects a tuple of (text, filepath)
    title="AI Meeting Assistant",
    description="Upload an audio file of a meeting. This tool will transcribe the audio, fix product-related terminology, and generate meeting minutes along with a list of tasks."
)

iface.launch(server_name='0.0.0.0', debug=True, allowed_paths=['/content/'])